In [ ]:
import sys
try:
    import pyLOM
except ImportError as e:
    print(f'Error importing pyLOM: {e}')
    print('Importing with local repository')
    sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO, SAM

def read_db_CODA(datafolder, case_idx):
    db = FRODO(root_dir = datafolder, format = 'CODA', initial_parse = True)
    
    db.extract_inputs(
        id_groups = (3,),
        cases_idx = case_idx,
        vtu_type='surface',
        verbose=False
        )

    for stage in [0, 1]:
        db.extract_outputs(
            id_groups=(3,),
            stage=stage, cases_idx = case_idx,
            var_name_excluded = [
                'BoundaryValues_CoefSkinFrictionX',
                'BoundaryValues_CoefSkinFrictionY',
                'BoundaryValues_CoefSkinFrictionZ'
                ],
            vtu_type='surface',
            )
    
    return db

def read_db_PYLOM(datafolder):
    db0 = FRODO(root_dir = datafolder, format = 'PYLOM', file = 'CADGroup_3_completo_stage_0.h5', initial_parse = True)

    db0.extract_inputs(
        keys_inputs={
            'ptos': 'xyz',    # coordenadas del mallado → data_dict['inputs']['ptos']
            'aoa': 'aoa',   # variable paramétrica   → data_dict['inputs']['aoa']
            'mach': 'M',   # variable paramétrica   → data_dict['inputs']['M']
        },
        keys_aux={},
    )

    db0.extract_outputs(
        keys_outputs={'cp': 'BoundaryValues_CoefPressure'}  # field del Dataset → data_dict['outputs']['cp']
    )

    db1 = FRODO(root_dir = datafolder, format = 'PYLOM', file = 'CADGroup_3_completo_stage_1.h5', initial_parse = True)

    db1.extract_inputs(
        keys_inputs={
            'ptos': 'xyz',    # coordenadas del mallado → data_dict['inputs']['ptos']
            'aoa': 'aoa',   # variable paramétrica   → data_dict['inputs']['aoa']
            'mach': 'M',   # variable paramétrica   → data_dict['inputs']['M']
        },
        keys_aux={},
    )

    db1.extract_outputs(
        keys_outputs={'cp': 'BoundaryValues_CoefPressure'}  # field del Dataset → data_dict['outputs']['cp']
    )

    print(db1.data_dict)

In [ ]:
from typing import Union
import os
def juntar_db(list_folders:Union[tuple[str], list[str]], case_idx:Union[tuple, list]):
    list_db = []
    
    for folder, cases in zip(list_folders, case_idx):
        db = read_db_CODA(datafolder = folder, case_idx=cases)
        
        if db.data_dict['CADGroup_3']['FlCc'].shape[-1] > 2:
            flcc = db.data_dict['CADGroup_3']['FlCc'][:, :-1]
            design_vars = db.metadata['design_vars']
            db.metadata['design_vars'] = design_vars[:-1]
            db.data_dict['CADGroup_3']['FlCc'] = flcc
    
        list_db.append(db)
        
    db_basic = FRODO.merge_datasets(
        root_dir='/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_completed',
        sources = [(db, '3') for db in list_db], #[(db_0, '3'), (db_1, '3'), (db_trans, '3')],
        new_group_id='3_completo',
        k=4,
        mesh_ref=0,
        cache=True,
        get_df_metrics_attr={
            'var_metrics': ['CoefLift', 'CoefDrag', 'CoefMomentY'],
            'iter_var': 1000,
            'save' : False
        }
        
    )
    
    return db_basic

# Base de datos original
# case_idx = list(range(100))
# fuera = [64, 79, 87, 88, 94]
# for c in fuera:
#     case_idx.remove(c)

case_idx = list(range(5))
db_completo = juntar_db(
    list_folders = [
        os.path.join('/home/m.jaraiz/Documentos/DATASETS/data_TIFON/', folder) for folder in ['rans3_basic', 'rans3_basic_rest', 'rans3_transonic_1']
        ],
    case_idx=(case_idx, 'all', case_idx)
)

In [ ]:
results = db_completo.stats.stage_difference_stats_variable(
    id_group='3_completo',
    stages=(0, 1),
    variable = 'BoundaryValues_CoefPressure',
    plots = {'violinplot': True, 'histogram': True, 'boxplot': True, 'scatter': True, 'histogram2D': False},
    kwargs_plots = {
        'histogram' : {'xscale': 'linear', 'yscale': 'log', 'bins': 20},
        'boxplot': {'marker_size': 0.5, "showfliers": True},
        'scatter': {
            "projection": "auto",      # auto | pca2 | pca3
            "scale_before_pca": True,
            "norm": "symlog",
            "linthresh": 1e-3,
            },
        "violinplot": {
            # Apariencia
            "fill": True,
            "linewidth": 1.5,
            "linecolor": "black",

            # Distribución
            "cut": 0,
            "bw_method": "scott",
            "bw_adjust": 1.0,
            "density_norm": "area",

            # Interior
            "inner": "box",          # box quart point stick None

            # Anchura
            "width": 0.9,
            "gridsize": 100,

            # Separación
            "gap": 0.05,

            # Escala
            "log_scale": False,

            # Color
            "color": "steelblue",

        },
        "histogram2D" : {

            "bins":40,

            "cmap":"viridis",

            "pthresh":0,

            "pmax":0.98,

            "log_scale":False,

        }
    }
    
)

## blxplot
# bajar grosor puntos de outlier
# count(outliers)
# pintar con y sin outliers

##violinplot
# mirar opción de quitar velas (puede que sean pocos pero me alargan las velas)

## histograma por bines (2D)
# en lugar de barras -> colores (seaborn)

In [ ]:
dict_stage_stats = db_completo.stats.stage_difference_stats(
    id_group='3_completo',
    stages=(0, 1),
    verbose=True    
)
SAM.DictVisualizer.rich_tree(dict_stage_stats)

In [ ]:
display(dict_stage_stats['0_vs_1']['global'])

In [ ]:
print(results)